# DiCo-NLI Track 1 — experiments for Assignment 3 (Colab)

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Not part of Assignment 2. These runs write to `results_exp/` and leave the frozen A2 results in `results_a2/` untouched.
Run cells 1 and 2 (setup, Drive), then the experiment cells.

In [ ]:
# 1. Setup: clone the repo (data, official scorer, src/data.py)
!git clone -q https://github.com/MuneebBaig71/DiCo-NLI_SemEval.git
%cd DiCo-NLI_SemEval
!pip -q install sentencepiece
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

In [ ]:
# 2. (Recommended) save results to Google Drive so a disconnect doesn't lose them
from google.colab import drive
drive.mount("/content/drive")
DRIVE_OUT = "/content/drive/MyDrive/DiCo_A2_results"

In [ ]:
# 3. English experiments: stronger starting points for the same pipeline.
#     E1 = DeBERTa-v3-base already fine-tuned on NLI data (MNLI/FEVER/ANLI), new 4-class head
#     E2 = DeBERTa-v3-large
#     E3 = DeBERTa-v3-large already fine-tuned on NLI data
#     All use the M3 recipe (consistency loss + Neg weight). Large models use lr 1e-5.
#     Time on a T4: about 3.5 min per base run, about 10-12 min per large run.
import importlib, json, shutil
from pathlib import Path
import src.train_a2; importlib.reload(src.train_a2)
from src.train_a2 import run

OUT, DRIVE_EXP = "results_exp", "/content/drive/MyDrive/DiCo_EXP_results"
if Path(DRIVE_EXP).exists():   # resume after a disconnect
    shutil.copytree(DRIVE_EXP, OUT, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("*.gsheet", "*.gdoc", "*.gslides", "*.gdraw"))

M3 = dict(lambda_cons=1.0, neg_weight=2.0, out_dir=OUT)
EXPS = [
    dict(tag="E1_nli_base",  model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli", reinit_head=True, **M3),
    dict(tag="E2_large",     model="microsoft/deberta-v3-large", lr=1e-5, **M3),
    dict(tag="E3_nli_large", model="MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli",
         reinit_head=True, lr=1e-5, **M3),
]
SEEDS = [13, 42, 87]   # start with [42] if you only want a quick look

for e in EXPS:
    for seed in SEEDS:
        tag = f"{e['tag']}_seed{seed}"
        if Path(f"{OUT}/runs/{tag}.json").exists():
            print("skip (done):", tag); continue
        try:
            run(**{**e, "tag": tag, "seed": seed})
        except Exception as err:      # one failed run should not stop the rest
            print(f"FAILED {tag}: {type(err).__name__}: {err}"); continue
        shutil.copytree(OUT, DRIVE_EXP, dirs_exist_ok=True)
print("done")

In [ ]:
# 4. Compare the experiments with the Assignment 2 systems (validation and dev, plus ensembles).
#     Bring the A2 runs back from Drive so they appear in the same table.
import shutil
from pathlib import Path
if not list(Path("results_a2/predictions").glob("M3_*_val_probs.csv")):
    shutil.copytree("/content/drive/MyDrive/DiCo_A2_results", "results_a2", dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("*.gsheet", "*.gdoc", "*.gslides", "*.gdraw"))
!python 08_exp_summary.py results_exp results_a2
shutil.copytree("results_exp", "/content/drive/MyDrive/DiCo_EXP_results", dirs_exist_ok=True)